In [1]:
import pandas as pd
import numpy as np
import ast

## Label name mapping

In [2]:
LABEL_NAMES = {
    0 : "amusement", 1 : "excitement", 2 : "joy",
    3 : "love", 4 : "desire", 5: "optimism", 
    6 : "caring", 7 : "pride", 8 : "admiration",
    9 : "gratitude",  10 : "relief", 11 : "approval",
    12 : "realization", 13 : "surprise", 14 : "curiosity",
    15 : "confusion", 16 : "fear", 17 : "nervousness",
    18 : "remorse", 19 : "embarrassment", 20 : "disappointment",
    21 : "sadness", 22 : "grief", 23 : "disgust",
    24 : "anger", 25 : "annoyance", 26 : "disapproval", 
    27 : "neutral"
}

## Load Data

In [3]:
from pathlib import Path

DATA_DIR = Path('../data/raw')

train = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8', encoding_errors='replace')
val   = pd.read_csv(DATA_DIR / 'val.csv',   encoding='utf-8', encoding_errors='replace')
test  = pd.read_csv(DATA_DIR / 'test.csv',  encoding='utf-8', encoding_errors='replace')


print(train.shape, val.shape, test.shape)

(16531, 3) (2066, 3) (2067, 3)


In [4]:
train.head(10)

,id,text,labels
0,tik000008,Xem mà ngẫm lại cuộc đời bản thân ta đã trải q...,[12]
1,5743,bức ảnh xuất sắc ❤️,"[2, 8, 3]"
2,32895,"Vừa đẹp trai, vừa tài giỏi. Nhà mặt phố, bố là...","[8, 7]"
3,you001182,"Bài học: <br>1: 5 nhìn, 4 chạm, 3 nghe, 2 ngửi...",[27]
4,12052,Dima Egiazarov bởi vì chúng tôi là người Việt ...,"[24, 23]"
5,1824,giờ mới biết,"[12, 13]"
6,tik023326,cảm thấy tự hào về đất nước mik,[7]
7,1046,nhìn mặt là cười phọt rồi,[0]
8,11556,Cris Minh Algeria nó theo đạo hồi. Ko đc chuyể...,[27]
9,2320,nó giống tao ghê . gặp tao tao cũng chọn mày,"[0, 11, 2, 3]"


## Parse cột labels thành list

In [5]:
train['labels'] = train['labels'].apply(ast.literal_eval)
val['labels']   = val['labels'].apply(ast.literal_eval)
test['labels']  = test['labels'].apply(ast.literal_eval)

print(type(train['labels'][0]))  # kiểm tra kiểu dữ liệu

<class 'list'>


## Label distribution

In [6]:
from collections import Counter

label_counts = Counter(label for labels in train['labels'] for label in labels)
pd.Series({LABEL_NAMES[k]: v for k, v in label_counts.most_common(10)}, name='count')

amusement         2868
sadness           2785
annoyance         2662
joy               1614
anger             1589
disappointment    1522
disgust           1206
love              1175
caring            1062
optimism          1046
Name: count, dtype: int64

amusement (2,868) chiếm gần gấp đôi các nhãn phổ biến khác. Phân bố không đồng đều rõ rệt ngay ở top 10.

### Nhãn ít nhất:

In [7]:
pd.Series({LABEL_NAMES[k]: v for k, v in label_counts.most_common()[-10:]}, name='count')

gratitude      765
desire         762
grief          747
fear           745
nervousness    728
remorse        707
realization    687
confusion      676
surprise       651
relief         635
Name: count, dtype: int64

relief (635) thấp hơn amusement 4.5 lần. Các nhãn thiểu số như disapproval, surprise, relief sẽ khó học với threshold 0.5 cố định, đây là bằng chứng cho RQ3.

### Trung bình số nhãn mỗi câu:

In [8]:
train['num_labels'] = train['labels'].apply(len)
print(train['num_labels'].describe())

count    16531.000000
mean         1.908233
std          0.790437
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max          5.000000
Name: num_labels, dtype: float64


Median: 2 nhãn/câu, max: 5 nhãn. Dataset thực sự multi-label thay vì single-label. Model phải học dự đoán nhiều cảm xúc đồng thời.

### Text analysis

In [9]:
train['text_length'] = train['text'].apply(len)
print(train['text_length'].describe())

count    16531.000000
mean        66.509407
std         66.533940
min          1.000000
25%         28.000000
50%         47.000000
75%         81.000000
max        906.000000
Name: text_length, dtype: float64


75% câu dưới 81 ký tự. Có thể dùng max_length=128 cho BERT thay vì 512 mặc định giúp tiết kiệm bộ nhớ và tăng tốc training đáng kể.

### Emoji analysis

In [10]:
import emoji

train['has_emoji'] = train['text'].apply(lambda x: bool(emoji.emoji_list(x)))
train['emoji_count'] = train['text'].apply(lambda x: len(emoji.emoji_list(x)))

train['has_emoji'].value_counts()

has_emoji
False    12422
True      4109
Name: count, dtype: int64

In [11]:
train.loc[train['has_emoji'], 'emoji_count'].describe()

count    4109.00000
mean        1.90825
std         2.18800
min         1.00000
25%         1.00000
50%         1.00000
75%         2.00000
max        48.00000
Name: emoji_count, dtype: float64

24.9% câu chứa emoji, chiếm gần 1/4 tập train. Trung bình 1.91 emoji/câu (trong số câu có emoji). Đủ lớn để chiến lược xử lý emoji ảnh hưởng đến kết quả model, bằng chứng trực tiếp cho RQ2.

### Teencode frequency

In [12]:
TEENCODE = ['k', 'ko', 'kg', 'kh', 'đc', 'dc', 'vs', 'mn', 'mk', 
            'mik', 'bn', 'bh', 'nx', 'cx', 'ck', 'vk', 'bt', 'tl',
            'rep', 'cmt', 'like', 'share', 'fb', 'ib', 'dm', 'vloz']

import re

def has_teencode(text):
    words = re.findall(r'\b\w+\b', text.lower())
    return any(w in TEENCODE for w in words)

train['has_teencode'] = train['text'].apply(has_teencode)

train['has_teencode'].value_counts()

has_teencode
False    13999
True      2532
Name: count, dtype: int64

In [13]:
train['has_teencode'].value_counts(normalize=True)

has_teencode
False    0.846833
True     0.153167
Name: proportion, dtype: float64

15.3% câu chứa teencode. Ít hơn emoji (24.9%) về tần suất, nhưng tác động lớn hơn về chất lượng vì teencode khiến BERT tokenize sai ngữ nghĩa hoàn toàn. Bằng chứng cho RQ1.

## SQL

### Flatten Data

In [22]:
import duckdb

rows = []
for _, r in train.iterrows():
    for lbl in r['labels']:
        rows.append({
            'label': lbl,
            'label_name': LABEL_NAMES[lbl],
            'has_emoji': int(r['has_emoji']),
            'has_teencode': int(r['has_teencode']),
            'text_length': r['text_length']
        })
df_flat = pd.DataFrame(rows)
print(f'Tổng số pairs: {len(df_flat):,}')

Tổng số pairs: 31,545


### Query 1: Window Function xeếp hạng theo tần suất

In [27]:
con = duckdb.connect()
con.register('df_flat', df_flat)

q1 = con.execute("""
    SELECT
        label_name,
        cnt,
        ROW_NUMBER() OVER (ORDER BY cnt DESC)            AS rank,
        ROUND(cnt * 100.0 / SUM(cnt) OVER (), 2)         AS pct,
        ROUND(MAX(cnt) OVER () * 1.0 / cnt, 2)           AS imbalance_vs_top
    FROM (
        SELECT label_name, COUNT(*) AS cnt
        FROM df_flat
        GROUP BY label_name
    )
    ORDER BY cnt DESC
""").df()
q1


,label_name,cnt,rank,pct,imbalance_vs_top
0,amusement,2868,1,9.09,1.00
1,sadness,2785,2,8.83,1.03
2,annoyance,2662,3,8.44,1.08
3,joy,1614,4,5.12,1.78
4,anger,1589,5,5.04,1.80
5,disappointment,1522,6,4.82,1.88
6,disgust,1206,7,3.82,2.38
7,love,1175,8,3.72,2.44
8,caring,1062,9,3.37,2.70
9,optimism,1046,10,3.32,2.74


Top 3 nhãn chiếm 26% toàn bộ data. Từ rank 12 trở đi, imbalance_vs_top vượt 3x nên model dùng threshold 0.5 cố định sẽ thiên về nhãn phổ biến và bỏ sót nhóm này. Đây là bằng chứng số cho RQ3

### Query 2: Phân tích tỉ lệ emoji và teencode theo từng nhãn, chia minority/majority

In [28]:
q2 = con.execute("""
    WITH label_stats AS (
        SELECT
            label_name,
            COUNT(*)                                          AS total,
            SUM(has_emoji)                                    AS emoji_cnt,
            ROUND(SUM(has_emoji) * 100.0 / COUNT(*), 1)      AS emoji_pct,
            SUM(has_teencode)                                 AS teencode_cnt,
            ROUND(SUM(has_teencode) * 100.0 / COUNT(*), 1)   AS teencode_pct
        FROM df_flat
        GROUP BY label_name
    ),
    median_cte AS (
        SELECT MEDIAN(total) AS med FROM label_stats
    )
    SELECT
        ls.label_name,
        ls.total,
        ls.emoji_pct,
        ls.teencode_pct,
        CASE WHEN ls.total < mc.med THEN 'minority' ELSE 'majority' END AS label_group
    FROM label_stats ls, median_cte mc
    ORDER BY ls.total
""").df()
q2


,label_name,total,emoji_pct,teencode_pct,label_group
0,relief,635,38.3,19.5,minority
1,surprise,651,19.0,8.3,minority
2,confusion,676,19.8,24.4,minority
3,realization,687,13.8,18.6,minority
4,remorse,707,33.5,26.7,minority
5,nervousness,728,23.6,29.9,minority
6,fear,745,23.4,18.7,minority
7,grief,747,35.1,18.3,minority
8,desire,762,25.2,17.2,minority
9,gratitude,765,28.4,11.2,minority


Emoji và teencode tập trung nhiều hơn ở nhóm nhãn thiểu số và cảm xúc tiêu cực/phức tạp. Xóa emoji sẽ ảnh hưởng nặng hơn lên love (48.8%) và pride (44.4%) so với neutral (7.1%), chiến lược xử lý không nên áp đồng nhất (RQ2).

Teencode cao ở embarrassment (32.3%), nervousness (29.9%), remorse (26.7%), nhóm này vừa ít data vừa noisy, chuẩn hóa teencode có thể cải thiện recall cho minority labels (RQ1+RQ3).